# Chest Disease Detection Baseline: MedCLIP Fine-tuning

This notebook implements a compact baseline for the chest disease multilabel task using the reference notebook strategy:

- MedCLIP ViT as the first-choice backbone.
- BiomedCLIP and timm DenseNet121 as fallbacks.
- Multilabel `BCEWithLogitsLoss` with clipped positive class weights.
- Validation threshold tuning for sample-average F1.
- Submission generation that preserves pre-filled rows in the official template.

Default run settings are intentionally modest. Override with environment variables such as `EPOCHS`, `BATCH`, `BACKBONE`, `IMG_SIZE`, and `WORKERS`.

In [ ]:
# Install dependencies. On Kaggle, enable Internet for first run.
import importlib.util
import os
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

required = {
    "medclip": "medclip",
    "open_clip": "open_clip_torch",
    "timm": "timm",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing, "scikit-learn", "matplotlib", "seaborn", "tqdm"])
else:
    print("Core dependencies already installed.")

In [ ]:
import glob
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)

SEED = int(os.environ.get("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)


def mean_column_auc(y_true, y_prob):
    aucs = []
    for c in range(y_true.shape[1]):
        if len(np.unique(y_true[:, c])) > 1:
            aucs.append(roc_auc_score(y_true[:, c], y_prob[:, c]))
    return float(np.mean(aucs)) if aucs else float("nan")


def samples_f1(y_true, y_bin):
    return f1_score(y_true, y_bin, average="samples", zero_division=0)


def to_binary(prob, thr):
    b = (prob >= thr).astype(np.int64)
    empty = b.sum(axis=1) == 0
    if empty.any():
        b[empty, prob[empty].argmax(axis=1)] = 1
    return b


def best_threshold(y_true, prob, lo=0.05, hi=0.60, step=0.01):
    ths = np.arange(lo, hi + 1e-9, step)
    f1s = np.array([samples_f1(y_true, to_binary(prob, t)) for t in ths])
    i = int(f1s.argmax())
    return float(ths[i]), float(f1s[i]), ths, f1s

In [ ]:
COMP_NAME = os.environ.get("COMP_NAME", "individual-test-chest-disease-detection")


def candidate_roots():
    roots = []
    roots += [Path("/kaggle/input") / COMP_NAME]
    roots += [Path(p) for p in glob.glob("/kaggle/input/*")]
    roots += [Path("dataset") / COMP_NAME, Path("dataset"), Path("comp_data"), Path(".")]
    roots += [Path(p) for p in glob.glob("dataset/*")]
    seen = set()
    for r in roots:
        key = str(r.resolve()) if r.exists() else str(r)
        if key not in seen:
            seen.add(key)
            yield r


def find_data_root():
    manual = os.environ.get("DATA_ROOT")
    if manual:
        r = Path(manual)
        if (r / "train.csv").exists():
            return r
        raise FileNotFoundError(f"DATA_ROOT does not contain train.csv: {r}")

    for r in candidate_roots():
        if (r / "train.csv").exists():
            return r

    zips = sorted(Path(".").glob("*.zip"), key=lambda p: p.stat().st_size, reverse=True)
    if zips:
        import zipfile
        out = Path("dataset") / COMP_NAME
        out.mkdir(parents=True, exist_ok=True)
        print("Extracting", zips[0], "to", out)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(out)
        if (out / "train.csv").exists():
            return out

    raise FileNotFoundError("Could not find train.csv. Set DATA_ROOT or place the competition files under dataset/.")


def find_images_dir(root):
    root = Path(root)
    for rel in ["images/images", "images", "."]:
        d = root / rel
        if list(d.glob("*.jpg")) or list(d.glob("*.png")):
            return d
    for d in root.rglob("*"):
        if d.is_dir() and (list(d.glob("*.jpg")) or list(d.glob("*.png"))):
            return d
    raise FileNotFoundError(f"No image directory found under {root}")


DATA_ROOT = find_data_root()
IMAGES_DIR = find_images_dir(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("IMAGES_DIR:", IMAGES_DIR)

In [ ]:
train = pd.read_csv(DATA_ROOT / "train.csv")
LABELS = [c for c in train.columns if c != "filename"]
N_CLASSES = len(LABELS)
assert N_CLASSES > 0, "No label columns found. Expected train.csv with filename plus disease labels."

train[LABELS] = train[LABELS].fillna(0).clip(0, 1).astype(np.float32)

image_paths = {}
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in IMAGES_DIR.glob(ext):
        image_paths[p.name] = p

train = train[train["filename"].isin(image_paths)].reset_index(drop=True)
assert len(train) > 0, "No train filenames match images on disk."

submission_candidates = [DATA_ROOT / "test_submission.csv", DATA_ROOT / "sample_submission.csv"]
submission_path = next((p for p in submission_candidates if p.exists()), None)
assert submission_path is not None, "Could not find test_submission.csv or sample_submission.csv."

SAMPLE_SUB = pd.read_csv(submission_path)
assert "filename" in SAMPLE_SUB.columns, "Submission template must contain filename column."
missing_cols = [c for c in LABELS if c not in SAMPLE_SUB.columns]
assert not missing_cols, f"Submission template is missing label columns: {missing_cols}"

PREDICT_MASK = SAMPLE_SUB[LABELS].isna().all(axis=1)
test_files = SAMPLE_SUB.loc[PREDICT_MASK, "filename"].tolist()

print(f"Labels ({N_CLASSES}):", LABELS)
print("Train rows:", len(train))
print("Submission template:", SAMPLE_SUB.shape)
print("Rows to predict:", len(test_files), "| pre-filled rows:", int((~PREDICT_MASK).sum()))
train.head()

In [ ]:
pos_rate = train[LABELS].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 4))
pos_rate.plot(kind="bar", ax=ax)
ax.set_ylabel("positive rate")
ax.set_title("Label positive rate")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 3))
train[LABELS].sum(axis=1).value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_xlabel("positive labels per image")
ax.set_ylabel("image count")
ax.set_title("Number of findings per image")
plt.tight_layout()
plt.show()

In [ ]:
IMG_SIZE = int(os.environ.get("IMG_SIZE", "224"))
MED_MEAN = [0.5862785803043838] * 3
MED_STD = [0.27950088968644304] * 3
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
OPENCLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
OPENCLIP_STD = [0.26862954, 0.26130258, 0.27577711]


def build_transforms(mean, std):
    train_tf = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.96, 1.04)),
        T.ColorJitter(brightness=0.12, contrast=0.12),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    eval_tf = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    return train_tf, eval_tf


class CXRDataset(Dataset):
    def __init__(self, files, labels, transform, image_map):
        self.files = list(files)
        self.labels = labels
        self.transform = transform
        self.image_map = image_map

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        image = Image.open(self.image_map[filename]).convert("RGB")
        image = self.transform(image)
        if self.labels is None:
            return image, filename
        return image, torch.tensor(self.labels[idx], dtype=torch.float32)

In [ ]:
BACKBONE = os.environ.get("BACKBONE", "medclip-vit")


def build_backbone(name_pref):
    if name_pref == "auto":
        order = ["medclip-vit", "biomedclip", "timm-densenet121"]
    else:
        order = [name_pref]
        if name_pref == "medclip-vit":
            order += ["biomedclip", "timm-densenet121"]

    errors = []
    for name in order:
        try:
            if name == "medclip-vit":
                from medclip import MedCLIPModel, MedCLIPVisionModelViT

                old_load = torch.load
                old_load_state_dict = torch.nn.Module.load_state_dict
                map_location = "cuda" if torch.cuda.is_available() else "cpu"

                def patched_load(*args, **kwargs):
                    kwargs.setdefault("map_location", map_location)
                    return old_load(*args, **kwargs)

                def patched_load_state_dict(self, state_dict, strict=True, *args, **kwargs):
                    return old_load_state_dict(self, state_dict, False, *args, **kwargs)

                torch.load = patched_load
                torch.nn.Module.load_state_dict = patched_load_state_dict
                try:
                    full_model = MedCLIPModel(vision_cls=MedCLIPVisionModelViT)
                    full_model.from_pretrained()
                finally:
                    torch.load = old_load
                    torch.nn.Module.load_state_dict = old_load_state_dict

                class MedCLIPEncoder(nn.Module):
                    def __init__(self, vision_model):
                        super().__init__()
                        self.vision_model = vision_model

                    def forward(self, x):
                        out = self.vision_model(pixel_values=x)
                        return out[0] if isinstance(out, (tuple, list)) else out

                enc = MedCLIPEncoder(full_model.vision_model)
                with torch.no_grad():
                    feat_dim = enc(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]
                print(f"Loaded MedCLIP ViT, feat_dim={feat_dim}")
                return enc, feat_dim, MED_MEAN, MED_STD, name

            if name == "biomedclip":
                import open_clip
                model, _, _ = open_clip.create_model_and_transforms(
                    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
                )

                class OpenCLIPVisionEncoder(nn.Module):
                    def __init__(self, visual):
                        super().__init__()
                        self.visual = visual

                    def forward(self, x):
                        return self.visual(x)

                enc = OpenCLIPVisionEncoder(model.visual)
                with torch.no_grad():
                    feat_dim = enc(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]
                print(f"Loaded BiomedCLIP, feat_dim={feat_dim}")
                return enc, feat_dim, OPENCLIP_MEAN, OPENCLIP_STD, name

            if name in {"timm-densenet121", "densenet121"}:
                import timm
                enc = timm.create_model("densenet121", pretrained=True, num_classes=0, global_pool="avg")
                print(f"Loaded timm densenet121, feat_dim={enc.num_features}")
                return enc, enc.num_features, IMAGENET_MEAN, IMAGENET_STD, name

            if name.startswith("timm:"):
                import timm
                model_name = name.split(":", 1)[1]
                enc = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
                print(f"Loaded timm {model_name}, feat_dim={enc.num_features}")
                return enc, enc.num_features, IMAGENET_MEAN, IMAGENET_STD, name

            raise ValueError(f"Unknown backbone: {name}")
        except Exception as exc:
            errors.append((name, repr(exc)))
            print(f"Backbone {name} failed: {type(exc).__name__}: {exc}")

    raise RuntimeError("No backbone loaded: " + str(errors))


class CXRClassifier(nn.Module):
    def __init__(self, encoder, feat_dim, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(feat_dim, n_classes))

    def forward(self, x):
        feat = self.encoder(x)
        if feat.ndim > 2:
            feat = feat.flatten(1)
        return self.head(feat)


encoder, FEAT_DIM, MEAN, STD, BACKBONE_USED = build_backbone(BACKBONE)
model = CXRClassifier(encoder, FEAT_DIM, N_CLASSES).to(DEVICE)
train_tf, eval_tf = build_transforms(MEAN, STD)
print("Backbone in use:", BACKBONE_USED)

In [ ]:
EPOCHS = int(os.environ.get("EPOCHS", "6"))
BATCH = int(os.environ.get("BATCH", "32"))
WORKERS = int(os.environ.get("WORKERS", "0" if DEVICE.type == "mps" else "2"))
VAL_SIZE = float(os.environ.get("VAL_SIZE", "0.10"))

all_y = train[LABELS].values.astype("float32")
label_count_bin = np.clip(all_y.sum(axis=1).astype(int), 0, 3)
unique, counts = np.unique(label_count_bin, return_counts=True)
stratify = label_count_bin if len(unique) > 1 and counts.min() >= 2 else None

tr_idx, va_idx = train_test_split(
    np.arange(len(train)),
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=stratify,
)

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    CXRDataset(train["filename"].values[tr_idx], all_y[tr_idx], train_tf, image_paths),
    batch_size=BATCH,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    CXRDataset(train["filename"].values[va_idx], all_y[va_idx], eval_tf, image_paths),
    batch_size=BATCH,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=pin_memory,
)

pos = all_y[tr_idx].sum(axis=0)
neg = len(tr_idx) - pos
pos_weight = np.clip(neg / np.clip(pos, 1, None), 1.0, 10.0)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": float(os.environ.get("ENCODER_LR", "1e-5"))},
    {"params": model.head.parameters(), "lr": float(os.environ.get("HEAD_LR", "1e-4"))},
], weight_decay=float(os.environ.get("WEIGHT_DECAY", "1e-4")))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Train size:", len(tr_idx), "| Val size:", len(va_idx))
print("Batch:", BATCH, "| Workers:", WORKERS, "| Epochs:", EPOCHS)

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    probs, targets = [], []
    for x, y in tqdm(loader, desc="validate", leave=False):
        x = x.to(DEVICE)
        logits = model(x)
        probs.append(torch.sigmoid(logits).float().cpu().numpy())
        targets.append(y.numpy())
    probs = np.concatenate(probs)
    targets = np.concatenate(targets)
    auc = mean_column_auc(targets, probs)
    thr, f1, _, _ = best_threshold(targets, probs)
    return auc, f1, thr, probs, targets


history = []
best_f1 = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen = 0
    started = time.time()
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")

    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            loss = criterion(model(x), y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x.size(0)
        seen += x.size(0)
        pbar.set_postfix(loss=f"{running_loss / max(seen, 1):.4f}")

    scheduler.step()
    val_auc, val_f1, val_thr, _, _ = evaluate(val_loader)
    history.append({"epoch": epoch, "loss": running_loss / len(tr_idx), "val_auc": val_auc, "val_f1": val_f1, "thr": val_thr})
    print(
        f"epoch {epoch}/{EPOCHS} | loss {running_loss / len(tr_idx):.4f} | "
        f"val_auc {val_auc:.4f} | val_sample_f1 {val_f1:.4f} | thr {val_thr:.2f} | {time.time() - started:.0f}s"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

hist_df = pd.DataFrame(history)
print("Best validation sample-F1:", best_f1)
hist_df

In [ ]:
val_auc, _, _, val_prob, val_true = evaluate(val_loader)
BEST_THR, BEST_F1, ths, f1s = best_threshold(val_true, val_prob)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ths, f1s)
ax.axvline(BEST_THR, color="red", linestyle="--", label=f"best = {BEST_THR:.2f}")
ax.set_xlabel("threshold")
ax.set_ylabel("sample-average F1")
ax.set_title("Validation threshold tuning")
ax.legend()
plt.tight_layout()
plt.show()

val_bin = to_binary(val_prob, BEST_THR)
per_class_f1 = pd.Series(
    {label: f1_score(val_true[:, i], val_bin[:, i], zero_division=0) for i, label in enumerate(LABELS)}
).sort_values()

print(f"Validation sample-average F1 = {BEST_F1:.4f} @ threshold {BEST_THR:.2f}")
print(f"Validation mean column AUC = {val_auc:.4f}")
per_class_f1.plot(kind="barh", figsize=(8, max(4, 0.35 * len(per_class_f1))), title="Per-class F1")
plt.tight_layout()
plt.show()
per_class_f1

In [ ]:
test_image_paths = dict(image_paths)
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in IMAGES_DIR.glob(ext):
        test_image_paths[p.name] = p

missing_test = [f for f in test_files if f not in test_image_paths]
assert not missing_test, f"Missing test images, first examples: {missing_test[:5]}"

test_loader = DataLoader(
    CXRDataset(test_files, None, eval_tf, test_image_paths),
    batch_size=BATCH,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=pin_memory,
)


@torch.no_grad()
def predict(loader):
    model.eval()
    probs, names = [], []
    for x, filenames in tqdm(loader, desc="predict"):
        x = x.to(DEVICE, non_blocking=True)
        probs.append(torch.sigmoid(model(x)).float().cpu().numpy())
        names.extend(list(filenames))
    if not probs:
        return np.zeros((0, N_CLASSES), dtype=np.float32), []
    return np.concatenate(probs), names


test_prob, pred_names = predict(test_loader)
pred_bin = to_binary(test_prob, BEST_THR) if len(pred_names) else np.zeros((0, N_CLASSES), dtype=np.int64)
pred_map = {name: row for name, row in zip(pred_names, pred_bin)}

submission = SAMPLE_SUB.copy()
for idx in submission.index[PREDICT_MASK]:
    submission.loc[idx, LABELS] = pred_map[submission.at[idx, "filename"]]

submission[LABELS] = submission[LABELS].astype(int)
assert list(submission.columns) == list(SAMPLE_SUB.columns)
assert submission[LABELS].isna().sum().sum() == 0
assert submission[LABELS].isin([0, 1]).all().all()

out_path = Path("submission.csv")
submission.to_csv(out_path, index=False)
print("Saved", out_path.resolve())
print(f"backbone={BACKBONE_USED} | val_sample_f1={BEST_F1:.4f} | threshold={BEST_THR:.2f}")
submission.head()